# 11. Model Explainability

After training and evaluating different machine learning models, we selected the best model based on performance metrics.

However, **how does the model make its predictions?** What features are most important? Why did it predict a particular groundwater level for a specific location and time?

This notebook focuses on **model interpretability and transparency**—understanding the reasoning behind the model's decisions rather than just its accuracy.

Model explainability is crucial because:

- **Trust**: Transparent predictions are easier to trust and defend
- **Debugging**: Understanding feature importance helps identify data quality issues
- **Insights**: Feature importance reveals which factors truly drive groundwater levels
- **Validation**: We can check if the model's reasoning aligns with hydrogeological knowledge
- **Actionability**: Knowing which features matter most guides data collection and monitoring priorities

In this notebook, we will:
1. Load the saved best model
2. Analyze feature importance
3. Generate SHAP explanations
4. Examine local predictions (specific examples)
5. Interpret the results in the context of groundwater science

## Step 1: Import Required Libraries

We import libraries for:
- Data manipulation: pandas, numpy
- Visualization: matplotlib, seaborn
- Model handling: joblib, pickle
- Explainability: SHAP, sklearn utilities

In [ ]:
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

try:
    import shap
    SHAP_AVAILABLE = True
except ImportError:
    SHAP_AVAILABLE = False
    print("Warning: SHAP library not available. Skipping SHAP analysis.")

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
from xgboost import XGBRegressor

warnings.filterwarnings("ignore")

# Plot settings
plt.style.use("default")
sns.set_palette("husl")

print("All required libraries imported successfully.")

All required libraries imported successfully.


## Step 2: Load the Feature-Engineered Dataset

We load the same feature-engineered dataset used for training.

This ensures that we perform explainability analysis on data with the same feature transformations applied.

In [2]:
candidate_paths = [
    Path.cwd().resolve() / "data" / "processed" / "groundwater_feature_engineered.csv",
    Path.cwd().resolve().parent / "data" / "processed" / "groundwater_feature_engineered.csv",
    Path("../data/processed/groundwater_feature_engineered.csv"),
    Path("data/processed/groundwater_feature_engineered.csv")
]

dataset_path = next((p for p in candidate_paths if p.exists()), None)
if dataset_path is None:
    raise FileNotFoundError("groundwater_feature_engineered.csv not found under data/processed/.")

df = pd.read_csv(dataset_path, parse_dates=["Data Acquisition Time"])

print(f"Dataset loaded from: {dataset_path}")
print(f"Dataset shape: {df.shape}")
print(f"\nColumn names:")
print(df.columns.tolist())
print(f"\nFirst few rows:")
df.head()

Dataset loaded from: /workspaces/groundwater-ai/data/processed/groundwater_feature_engineered.csv
Dataset shape: (69404, 25)

Column names:
['Station', 'Data Acquisition Time', 'Latitude', 'Longitude', 'Groundwater Level Telemetry 6 Hourly (meter)', 'RL_MSL', 'time_diff', 'Year', 'Month', 'Day', 'Hour', 'DayOfWeek', 'WeekOfYear', 'Quarter', 'IsWeekend', 'Lag_1', 'Lag_4', 'Lag_28', 'RollingMean_4', 'RollingStd_4', 'Hour_sin', 'Hour_cos', 'Month_sin', 'Month_cos', 'Station_ID']

First few rows:


,Station,Data Acquisition Time,Latitude,Longitude,Groundwater Level Telemetry 6 Hourly (meter),RL_MSL,time_diff,Year,Month,Day,...,Lag_1,Lag_4,Lag_28,RollingMean_4,RollingStd_4,Hour_sin,Hour_cos,Month_sin,Month_cos,Station_ID
0,Adakamaranahalli,2021-10-06 00:00:00,13.07,77.45,-22.50,849.0,1 days 06:00:00,2021,10,6,...,-22.71,-22.34,-22.51,-22.8625,0.419235,0.000000e+00,1.000000e+00,-1.0,-1.836970e-16,0
1,Adakamaranahalli,2021-10-06 06:00:00,13.07,77.45,-22.83,849.0,0 days 06:00:00,2021,10,6,...,-22.50,-23.16,-22.87,-22.9025,0.355563,1.000000e+00,6.123234e-17,-1.0,-1.836970e-16,0
2,Adakamaranahalli,2021-10-06 12:00:00,13.07,77.45,-22.88,849.0,0 days 06:00:00,2021,10,6,...,-22.83,-23.24,-23.23,-22.8200,0.311448,1.224647e-16,-1.000000e+00,-1.0,-1.836970e-16,0
3,Adakamaranahalli,2021-10-06 18:00:00,13.07,77.45,-22.52,849.0,0 days 06:00:00,2021,10,6,...,-22.88,-22.71,-22.66,-22.7300,0.169115,-1.000000e+00,-1.836970e-16,-1.0,-1.836970e-16,0
4,Adakamaranahalli,2021-10-07 00:00:00,13.07,77.45,-22.35,849.0,0 days 06:00:00,2021,10,7,...,-22.52,-22.50,-22.46,-22.6825,0.200395,0.000000e+00,1.000000e+00,-1.0,-1.836970e-16,0


## Step 3: Load the Saved Model and Metadata

We load:
- The trained best model from `models/best_model.pkl`
- Metadata from `models/best_model_meta.json` which contains the model type and feature list

This ensures we use exactly the same model and features that were selected during the model comparison phase.

In [3]:
project_root = Path.cwd().resolve()
model_dir = next(
    (candidate / "models" for candidate in [project_root, project_root.parent] if (candidate / "models").exists()),
    project_root / "models"
)

best_model_path = model_dir / "best_model.pkl"
meta_path = model_dir / "best_model_meta.json"

if not best_model_path.exists():
    raise FileNotFoundError(f"Best model not found at {best_model_path}")
if not meta_path.exists():
    raise FileNotFoundError(f"Model metadata not found at {meta_path}")

best_model = joblib.load(best_model_path)

with open(meta_path, "r") as f:
    model_meta = json.load(f)

print(f"Model loaded from: {best_model_path}")
print(f"\nModel Information:")
print(f"Model Type: {model_meta['model_name']}")
print(f"Number of Features: {len(model_meta['features'])}")
print(f"\nFeatures used for training:")
for i, feat in enumerate(model_meta['features'], 1):
    print(f"{i:2d}. {feat}")

Model loaded from: /workspaces/groundwater-ai/models/best_model.pkl

Model Information:
Model Type: Linear Regression
Number of Features: 21

Features used for training:
 1. Latitude
 2. Longitude
 3. RL_MSL
 4. Year
 5. Month
 6. Day
 7. Hour
 8. DayOfWeek
 9. WeekOfYear
10. Quarter
11. IsWeekend
12. Lag_1
13. Lag_4
14. Lag_28
15. RollingMean_4
16. RollingStd_4
17. Hour_sin
18. Hour_cos
19. Month_sin
20. Month_cos
21. Station_ID


## Step 4: Prepare the Feature Matrix

We extract the feature matrix **in the exact order** used during training.

Important: We must NOT include the target variable (Groundwater Level) as an input feature, as that would leak the answer into the explainability analysis.

In [4]:
target = "Groundwater Level Telemetry 6 Hourly (meter)"
features = model_meta['features']

# Prepare feature matrix
X = df[features].copy()
y = df[target].copy()

print(f"Feature matrix shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"\nFeature matrix info:")
print(f"Missing values: {X.isnull().sum().sum()}")
print(f"Data type: {X.dtypes.unique()}")
print(f"\nFeature statistics:")
X.describe()

Feature matrix shape: (69404, 21)
Target shape: (69404,)

Feature matrix info:
Missing values: 0
Data type: [dtype('float64') dtype('int64')]

Feature statistics:


,Latitude,Longitude,RL_MSL,Year,Month,Day,Hour,DayOfWeek,WeekOfYear,Quarter,...,Lag_1,Lag_4,Lag_28,RollingMean_4,RollingStd_4,Hour_sin,Hour_cos,Month_sin,Month_cos,Station_ID
count,69404.000000,69404.000000,69404.000000,69404.000000,69404.000000,69404.000000,69404.000000,69404.000000,69404.000000,69404.000000,...,69404.000000,69404.000000,69404.000000,69404.000000,69404.000000,6.940400e+04,6.940400e+04,69404.000000,6.940400e+04,69404.000000
mean,12.957263,77.581213,874.984814,2023.528053,6.697510,15.486067,9.001009,2.997608,27.326898,2.561409,...,-26.847169,-26.852121,-26.902423,-26.849442,2.516167,9.390868e-05,-3.160027e-04,-0.020959,4.138176e-02,11.119474
std,0.106524,0.123850,29.156135,1.132584,3.536492,8.677346,6.707159,1.968784,15.441294,1.149159,...,40.551514,40.542260,40.477655,39.808521,8.531377,7.070725e-01,7.071512e-01,0.709972,7.027111e-01,7.708363
min,12.707778,77.360000,800.000000,2021.000000,1.000000,1.000000,0.000000,0.000000,1.000000,1.000000,...,-190.106000,-190.106000,-190.106000,-189.991250,0.000000,-1.000000e+00,-1.000000e+00,-1.000000,-1.000000e+00,0.000000
25%,12.876944,77.500000,874.000000,2023.000000,4.000000,8.000000,6.000000,1.000000,13.000000,2.000000,...,-26.035250,-26.037000,-26.087000,-26.075250,0.017623,0.000000e+00,-1.000000e+00,-0.866025,-5.000000e-01,4.000000
50%,12.980000,77.540000,880.000000,2024.000000,7.000000,15.000000,12.000000,3.000000,27.000000,3.000000,...,-19.047500,-19.049000,-19.081000,-18.585125,0.071356,1.224647e-16,-1.836970e-16,0.000000,6.123234e-17,11.000000
75%,13.050000,77.695833,888.000000,2024.000000,10.000000,23.000000,12.000000,5.000000,42.000000,4.000000,...,-12.664000,-12.664000,-12.694000,-13.059062,0.594093,1.000000e+00,6.123234e-17,0.500000,8.660254e-01,19.000000
max,13.136111,77.790000,927.000000,2025.000000,12.000000,31.000000,18.000000,6.000000,52.000000,4.000000,...,79.026000,79.026000,79.026000,79.002750,109.940273,1.000000e+00,1.000000e+00,1.000000,1.000000e+00,23.000000


## Step 5: Feature Importance Analysis

Feature importance tells us which input variables have the strongest influence on the model's predictions.

For **Linear Regression** models, feature importance is measured by the **coefficients**—larger absolute values indicate stronger influence.

We will:
1. Extract the model coefficients
2. Sort them by absolute value (positive or negative)
3. Visualize them in a horizontal bar chart for easy interpretation


In [ ]:
model_type = model_meta['model_name']

if model_type == "Linear Regression":
    # Linear Regression: coefficients represent feature importance
    coefficients = best_model.coef_
    feature_importance = pd.DataFrame({
        "Feature": features,
        "Coefficient": coefficients,
        "Abs_Coefficient": np.abs(coefficients)
    }).sort_values("Abs_Coefficient", ascending=False)
    
    print("Model Type: Linear Regression")
    print(f"Intercept: {best_model.intercept_:.6f}")
    print(f"\nTop 10 Most Important Features (by Coefficient Magnitude):")
    print(feature_importance.head(10).to_string(index=False))
    
elif model_type == "Random Forest":
    # Random Forest: feature_importances_
    importances = best_model.feature_importances_
    feature_importance = pd.DataFrame({
        "Feature": features,
        "Importance": importances
    }).sort_values("Importance", ascending=False)
    
    print("Model Type: Random Forest")
    print(f"\nTop 10 Most Important Features:")
    print(feature_importance.head(10).to_string(index=False))
    
elif model_type == "XGBoost":
    # XGBoost: get_booster().get_score()
    importance_dict = best_model.get_booster().get_score(importance_type="weight")
    feature_importance = pd.DataFrame({
        "Feature": list(importance_dict.keys()),
        "Importance": list(importance_dict.values())
    }).sort_values("Importance", ascending=False)
    
    print("Model Type: XGBoost")
    print(f"\nTop 10 Most Important Features:")
    print(feature_importance.head(10).to_string(index=False))
    
else:
    print(f"Model type '{model_type}' not directly supported for feature importance extraction.")
    feature_importance = None

# Create output directory for explainability artifacts
project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent

output_dir = project_root / "outputs" / "explainability"
output_dir.mkdir(parents=True, exist_ok=True)
print(f"\nExplainability outputs will be saved to: {output_dir}")

if feature_importance is not None:
    feature_importance.to_csv(output_dir / "feature_importance.csv", index=False)
    print(f"Feature importance table saved to: {output_dir / 'feature_importance.csv'}")


### Visualization: Feature Importance Bar Chart

We create a horizontal bar chart showing the top features.

This visualization makes it easy to see at a glance which features the model relies on most.


In [ ]:
if feature_importance is not None:
    top_n = min(15, len(feature_importance))
    top_features = feature_importance.head(top_n)
    
    plt.figure(figsize=(10, 6))
    
    if model_type == "Linear Regression":
        colors = ['green' if x > 0 else 'red' for x in top_features['Coefficient']]
        plt.barh(range(len(top_features)), top_features['Coefficient'], color=colors, alpha=0.7)
        plt.xlabel('Coefficient Value')
        title_suffix = "(Linear Regression Coefficients)"
    else:
        plt.barh(range(len(top_features)), top_features.iloc[:, 1], color="steelblue", alpha=0.7)
        plt.xlabel('Importance Score')
        title_suffix = f"({model_type})"
    
    plt.yticks(range(len(top_features)), top_features['Feature'])
    plt.title(f'Top {top_n} Most Important Features {title_suffix}')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.savefig(output_dir / "feature_importance.png", dpi=300, bbox_inches="tight")
    plt.show()
    
    print(f"\n✓ Feature importance visualization created and saved to {output_dir / 'feature_importance.png'}.")


## Permutation Feature Importance

Permutation importance is a model-agnostic way to measure how much a feature affects prediction quality. The idea is simple: shuffle one feature at a time while keeping the rest unchanged, then measure how much the model performance drops. A large drop indicates that the feature is important.

This method is useful because it does not rely on model coefficients or tree-based feature importances, which can be difficult to interpret or sometimes misleading. It also works well as a second validation step alongside the built-in importance measures and SHAP values.

In this notebook, we use permutation importance to check whether the features identified by the model are genuinely influential for prediction quality.


In [ ]:
if feature_importance is not None:
    try:
        perm_result = permutation_importance(
            best_model,
            X,
            y,
            n_repeats=10,
            random_state=42,
            n_jobs=-1
        )
        permutation_importance_df = pd.DataFrame({
            "Feature": features,
            "Importance_Mean": perm_result.importances_mean,
            "Importance_STD": perm_result.importances_std
        }).sort_values("Importance_Mean", ascending=False)
        permutation_importance_df.to_csv(output_dir / "permutation_importance.csv", index=False)

        print("Permutation Feature Importance (Top 10):")
        print(permutation_importance_df.head(10).to_string(index=False))

        top_perm = permutation_importance_df.head(15).copy()
        plt.figure(figsize=(10, 6))
        plt.barh(top_perm["Feature"][::-1], top_perm["Importance_Mean"][::-1], color="darkorange", alpha=0.8)
        plt.xlabel("Mean Decrease in Model Score")
        plt.ylabel("Feature")
        plt.title("Permutation Feature Importance")
        plt.gca().invert_yaxis()
        plt.tight_layout()
        plt.savefig(output_dir / "permutation_importance.png", dpi=300, bbox_inches="tight")
        plt.show()
        print(f"Permutation importance table and chart saved to {output_dir / 'permutation_importance.csv'} and {output_dir / 'permutation_importance.png'}.")
    except Exception as e:
        print(f"Permutation importance could not be calculated: {e}")
else:
    print("Permutation importance not available because feature importance could not be extracted.")


## Step 6: SHAP Explainability

SHAP (SHapley Additive exPlanations) is a powerful technique for explaining individual predictions.

SHAP values show **how much each feature contributes** to pushing the prediction away from the model's average prediction.

To keep the analysis computationally efficient without losing interpretability, we do not calculate SHAP values on the full dataset. Instead, we use a reproducible random sample of at most 500 rows:

X_shap = X.sample(n=min(500, len(X)), random_state=42)

This makes the SHAP workflow much faster while still preserving a representative view of feature effects across the dataset. The sampled explanation remains stable and interpretable for both global and local analysis.

We will create:
1. **SHAP Summary Plot**: Shows the impact of each feature on predictions across the sample
2. **SHAP Bar Plot**: Shows average absolute feature importance
3. **SHAP Waterfall Plot**: Explains one local prediction in detail

Note: If SHAP is not supported for this model, we will display a clear message instead of crashing.


In [ ]:
X_shap = X.sample(n=min(500, len(X)), random_state=42)
print(f"Using SHAP sample with {len(X_shap)} rows from a total of {len(X)} rows.")

if not SHAP_AVAILABLE:
    print("SHAP library is not installed. Skipping SHAP analysis.")
    print("To enable SHAP, install it with: pip install shap")
else:
    try:
        print("Generating SHAP explanations on the sampled feature matrix...")
        
        if model_type in ["Linear Regression", "Random Forest", "XGBoost"]:
            explainer = shap.Explainer(best_model, X_shap)
            shap_values = explainer(X_shap)
            
            print("✓ SHAP explainer created successfully on the sampled dataset.")
            print("\nSHAP analysis complete.")
        else:
            print(f"SHAP may not be optimized for model type: {model_type}")
            print("Attempting generic SHAP explainer...")
            try:
                explainer = shap.Explainer(best_model, X_shap)
                shap_values = explainer(X_shap)
                print("✓ Generic SHAP explainer created.")
            except Exception as e:
                print(f"SHAP not supported for this model: {str(e)}")
                shap_values = None
    except Exception as e:
        print(f"Error creating SHAP explanations: {str(e)}")
        print("Continuing with other analysis methods...")
        shap_values = None


### SHAP Summary Plot

The summary plot shows:
- Each row is a feature
- Each point represents a sample
- Red points = high feature values, Blue points = low feature values
- Position on x-axis = impact on prediction

**Interpretation**:
- Features at the top have the highest impact on predictions
- Red dots pushed right = high feature values increase prediction
- Blue dots pushed left = low feature values increase prediction


In [ ]:
if SHAP_AVAILABLE and shap_values is not None:
    try:
        plt.figure(figsize=(12, 8))
        shap.summary_plot(shap_values, X_shap, show=False)
        plt.title('SHAP Summary Plot: Feature Impact on Model Predictions')
        plt.tight_layout()
        plt.savefig(output_dir / "shap_summary.png", dpi=300, bbox_inches="tight")
        plt.show()
        print("\n✓ SHAP Summary Plot created successfully and saved to the outputs folder.")
    except Exception as e:
        print(f"Could not create SHAP summary plot: {str(e)}")
else:
    print("Skipping SHAP summary plot (SHAP not available or not supported).")


### SHAP Bar Plot

The bar plot shows the mean absolute SHAP value for each feature.

This is similar to feature importance—features with higher bars have stronger average impact on predictions.


In [ ]:
if SHAP_AVAILABLE and shap_values is not None:
    try:
        plt.figure(figsize=(10, 6))
        shap.summary_plot(shap_values, X_shap, plot_type="bar", show=False)
        plt.title('SHAP Bar Plot: Average Impact of Features')
        plt.tight_layout()
        plt.savefig(output_dir / "shap_bar.png", dpi=300, bbox_inches="tight")
        plt.show()
        print("\n✓ SHAP Bar Plot created successfully and saved to the outputs folder.")
    except Exception as e:
        print(f"Could not create SHAP bar plot: {str(e)}")
else:
    print("Skipping SHAP bar plot (SHAP not available or not supported).")


## Step 7: Local Prediction Explanation

So far, we've analyzed global feature importance (across the entire dataset).

Now we examine a **single prediction** to understand why the model made that specific decision.

We will:
1. Select one reproducible random sample from the dataset
2. Show its actual groundwater level
3. Show the model's prediction
4. Explain which features pushed the prediction up or down
5. Create a SHAP waterfall plot (if available) showing the contribution of each feature


In [ ]:
# Select a reproducible random sample for local explanation
rng = np.random.RandomState(42)
sample_idx = int(rng.randint(0, len(X)))

X_sample = X.iloc[[sample_idx]].copy()
y_sample_actual = y.iloc[sample_idx]
y_sample_pred = best_model.predict(X_sample)[0]

print(f"Selected Sample Index: {sample_idx}")
print(f"\nActual Groundwater Level: {y_sample_actual:.3f} m")
print(f"Predicted Groundwater Level: {y_sample_pred:.3f} m")
print(f"Prediction Error: {abs(y_sample_actual - y_sample_pred):.3f} m")
print(f"\nSample Features:")
sample_df = pd.DataFrame({
    "Feature": features,
    "Value": X.iloc[sample_idx].values
})
print(sample_df.to_string(index=False))


### Feature Contribution to This Prediction

For a Linear Regression model, we can calculate how much each feature contributes to the prediction:

Prediction = Intercept + (Feature₁ × Coefficient₁) + (Feature₂ × Coefficient₂) + ...

We will show the top contributing features (positive and negative) for this sample.


In [ ]:
if model_type == "Linear Regression":
    X_sample_values = X.iloc[sample_idx].values
    contributions = X_sample_values * best_model.coef_
    
    contribution_df = pd.DataFrame({
        "Feature": features,
        "Coefficient": best_model.coef_,
        "Feature_Value": X_sample_values,
        "Contribution": contributions
    }).sort_values("Contribution", key=lambda s: s.abs(), ascending=False)
    
    print("\nTop Features Contributing to This Prediction:")
    print("\n(Positive values push prediction UP, Negative values push it DOWN)\n")
    print(contribution_df.head(10).to_string(index=False))
    
    print(f"\n\nPrediction Breakdown:")
    print(f"Intercept: {best_model.intercept_:.6f}")
    print(f"Sum of Feature Contributions: {contributions.sum():.6f}")
    print(f"Final Prediction: {y_sample_pred:.6f}")
else:
    print(f"\nFeature contribution breakdown not directly available for {model_type}.")
    print("Use SHAP waterfall plot below for local explanation.")


### SHAP Waterfall Plot for This Sample

A waterfall plot shows how each feature's SHAP value contributes to moving the prediction away from the base (average) prediction.

Red bars = push prediction up (increase groundwater level estimate)

Blue bars = push prediction down (decrease groundwater level estimate)


In [ ]:
if SHAP_AVAILABLE and shap_values is not None:
    try:
        sample_shap_idx = int(np.random.RandomState(42).randint(0, len(X_shap)))
        if sample_shap_idx >= len(shap_values):
            sample_shap_idx = len(shap_values) - 1
        
        plt.figure(figsize=(12, 6))
        shap.waterfall_plot(shap_values[sample_shap_idx], show=False)
        plt.title(f'SHAP Waterfall Plot: Feature Contributions for Sample {sample_idx}')
        plt.tight_layout()
        plt.savefig(output_dir / "shap_waterfall.png", dpi=300, bbox_inches="tight")
        plt.show()
        print("\n✓ SHAP Waterfall Plot created successfully and saved to the outputs folder.")
    except Exception as e:
        print(f"Could not create SHAP waterfall plot: {str(e)}")
else:
    print("Skipping SHAP waterfall plot (SHAP not available or not supported).")


## Step 8: Key Findings and Interpretation

The final interpretability step is to compare the actual model outputs rather than rely on generic assumptions. The results from this notebook show which features appear in the most important positions and how these factors relate to groundwater behaviour.

We will interpret the ranking produced by the model, focusing on the actual variables that emerge in the top 10 and the physical meaning of those features.


In [ ]:
if feature_importance is not None:
    print("="*60)
    print("TOP 10 MOST IMPORTANT FEATURES")
    print("="*60)
    
    top_10 = feature_importance.head(10).copy()
    top_10['Rank'] = range(1, 11)
    
    if model_type == "Linear Regression":
        print(top_10[['Rank', 'Feature', 'Coefficient', 'Abs_Coefficient']].to_string(index=False))
    else:
        print(top_10[['Rank', 'Feature', 'Importance']].to_string(index=False))
    
    top_10_features = top_10['Feature'].tolist()
    print("\nActual top-10 feature list:")
    print(", ".join(top_10_features))
    
    print("\n" + "="*60)
    print("RESULT-BASED INTERPRETATION")
    print("="*60)
    
    if "Lag_1" in top_10_features:
        print("- Lag_1 appears among the strongest predictors, which indicates that recent groundwater levels carry substantial predictive signal. This is consistent with groundwater systems that respond gradually to previous conditions and short-term persistence.")
    if "RollingMean_4" in top_10_features:
        print("- RollingMean_4 is influential, which suggests that the model is learning short-term trend behaviour rather than responding only to isolated spikes. This fits groundwater dynamics where recent averages reflect aquifer storage and response time.")
    if "RL_MSL" in top_10_features:
        print("- RL_MSL is present in the important feature set, indicating that elevation and local ground reference conditions contribute to prediction differences between stations. Hydrological behaviour is often spatially conditioned by topography and groundwater head.")
    if "Hour" in top_10_features:
        print("- Hour is relevant in the top features, which implies that temporal conditions such as daily cycles or observational timing influence the model. This may reflect periodic measurement patterns or time-varying hydroclimatic conditions.")
    if "Month" in top_10_features or "DayOfWeek" in top_10_features or "Quarter" in top_10_features:
        print("- Seasonal or time-based variables contribute to the model, suggesting that groundwater behaviour varies across the year and is not driven only by recent levels.")
    if "Latitude" in top_10_features or "Longitude" in top_10_features:
        print("- Spatial coordinates are important, which indicates that location-specific hydrogeological conditions are being captured by the model.")
    
    if not top_10_features:
        print("- No top-ranking features were identified in the current model output.")
    print("\nThese findings show that the model is not relying on arbitrary features alone; it is giving weight to variables that are physically and temporally meaningful for groundwater behaviour.")
else:
    print("Feature importance not available for this model type.")


## Step 9: Hydrogeological Interpretation

Based on the feature importance results, we interpret the model's behavior in the context of groundwater science.

Key observations to look for:
- **Lag features dominate?** → Model relies on historical groundwater levels (expected)
- **Rolling statistics important?** → Model captures short-term variability
- **Temporal features matter?** → Seasonality influences groundwater levels
- **Spatial features influential?** → Location and station characteristics matter
- **Cyclical features contribute?** → Daily and monthly cycles present in the data

This helps validate whether the model has learned physically meaningful patterns or if there are data quality issues.


In [ ]:
print("\n" + "="*60)
print("GROUNDWATER INTERPRETATION")
print("="*60 + "\n")

if feature_importance is not None:
    top_10_features = feature_importance.head(10)['Feature'].tolist()
    print(f"Top 10 features identified by the model: {', '.join(top_10_features)}")

    lag_features_top = [f for f in top_10_features if f in ['Lag_1', 'Lag_4', 'Lag_28']]
    rolling_features_top = [f for f in top_10_features if f in ['RollingMean_4', 'RollingStd_4']]
    temporal_features_top = [f for f in top_10_features if f in ['Year', 'Month', 'Day', 'Hour', 'DayOfWeek', 'WeekOfYear', 'Quarter', 'IsWeekend', 'Hour_sin', 'Hour_cos', 'Month_sin', 'Month_cos']]
    spatial_features_top = [f for f in top_10_features if f in ['Latitude', 'Longitude', 'RL_MSL', 'Station_ID']]

    if lag_features_top:
        print(f"- The presence of {', '.join(lag_features_top)} among the strongest variables suggests that recent groundwater conditions are highly predictive, which is consistent with temporal persistence in aquifer storage.")
    if rolling_features_top:
        print(f"- The inclusion of {', '.join(rolling_features_top)} indicates that short-term trend and variability are relevant to the model's prediction, reinforcing the importance of recent hydrological conditions.")
    if temporal_features_top:
        print(f"- The model also weights temporal features such as {', '.join(temporal_features_top)}, indicating that groundwater levels vary meaningfully across time periods and seasonal conditions.")
    if spatial_features_top:
        print(f"- Spatial variables such as {', '.join(spatial_features_top)} highlight that the groundwater response is not identical across locations, which reflects local hydrogeological and topographic differences.")

    if not (lag_features_top or rolling_features_top or temporal_features_top or spatial_features_top):
        print("- The current model output does not show a strong dominance of any single category, which suggests that the prediction depends on a more balanced combination of variables.")
else:
    print("- Feature importance is not available for the selected model type, so hydrogeological interpretation is limited to the available explainability outputs.")


## Step 10: Summary

### What We Learned

In this notebook, we examined **how the trained groundwater prediction model makes its decisions**:

1. **Global Feature Importance**: We identified which input variables have the strongest influence on predictions using model-specific importance measures and permutation importance.

2. **SHAP Analysis**: We used SHAP values to understand the contribution of each feature across a representative sample of the dataset and for local predictions.

3. **Local Explanations**: We examined a specific prediction and explained which features pushed it up or down.

4. **Hydrogeological Validation**: We checked whether the model's reasoning aligns with groundwater behaviour and known physical patterns.

### Why Model Explainability Matters

**Trust**: A transparent model is easier to trust. Knowing *why* a prediction was made increases confidence in the result.

**Debugging**: If the model relies on unexpected features, it alerts us to possible data issues or modelling problems.

**Actionability**: Understanding which features matter most helps guide monitoring priorities, data collection, and interpretation of groundwater responses.

**Validation**: We can check whether the model's reasoning matches domain knowledge and hydrogeological understanding.

### Complementary to Model Evaluation

- **Notebook 07 (Model Evaluation)** showed the model's predictive performance.
- **This notebook (Explainability)** explains which variables are driving that performance and why the model makes certain decisions.

Together, these results provide stronger evidence that the model is not only accurate but also interpretable. The model appears to be learning from physically meaningful trends, including recent groundwater history, short-term variation, temporal structure, and spatial setting.

### Final Interpretation

The explainability analysis supports the view that the groundwater model is using signal that is consistent with the expected dynamics of the aquifer system. The strongest predictors are not random; they reflect recent conditions, temporal structure, and site-specific characteristics. This increases confidence that the model is capturing realistic groundwater behaviour rather than overfitting to noise.


---

This explainability notebook used several complementary tools to improve confidence in the groundwater prediction model. The built-in feature importance analysis showed the strongest predictors in the selected model, while permutation importance provided a model-agnostic check that the same variables remain influential when prediction quality is disturbed. SHAP analysis then added local and global insight into how each feature pushes predictions up or down.

The results indicate that recent groundwater conditions, short-term trend behaviour, and time- and location-specific variables are important contributors to model performance. These findings are consistent with groundwater systems that exhibit persistence, seasonal variation, and spatial heterogeneity. This makes the final prediction framework more transparent and easier to defend in an academic project context.

Together with the model evaluation notebook, this explainability analysis supports a stronger conclusion: the model is accurate, interpretable, and aligned with the physical behaviour of the groundwater system rather than simply memorising patterns in the data.
